<a href="https://colab.research.google.com/github/adriano-klein/clearbank-analise/blob/main/desafio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio prático: Análise Financeira com Python

## Funções do desafio



###Ler transações

In [ ]:
import csv

def ler_transacoes(caminho_arquivo):
    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)

    except FileNotFoundError:
        print(f"Erro: o arquivo '{caminho_arquivo}' não foi encontrado.")
        print("Verifique se o upload do arquivo foi realizado corretamente na sessão do Colab.")
        return None

###Validar transações

In [ ]:
from datetime import datetime

def validar_transacao(linha):
    id_bruto = linha.get("id", "").strip()
    cliente_id = linha.get("cliente_id", "").strip()
    data_bruta = linha.get("data", "").strip()
    tipo = linha.get("tipo", "").strip()
    valor_bruto = linha.get("valor", "").strip()
    descricao = linha.get("descricao", "").strip()
    categoria = linha.get("categoria", "").strip()

    if not id_bruto or not id_bruto.isdigit():
        return None

    if not cliente_id:
        return None

    if tipo not in ["credito", "debito"]:
        return None

    try:
        valor_convertido = float(valor_bruto)
        if valor_convertido <= 0:
            return None
    except (ValueError, TypeError):
        return None

    try:
        data_convertida = datetime.strptime(data_bruta, "%Y-%m-%d")
    except ValueError:
        return None

    return {
        "id": int(id_bruto),
        "data": data_convertida,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor_convertido,
        "descricao": descricao,
        "categoria": categoria
    }

###Transações suspeitas

In [ ]:
LIMITE_SUSPEITO = 10000
def transacoes_suspeitas(transacoes):
    suspeitas = [t for t in transacoes if t["valor"] > LIMITE_SUSPEITO]

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
        return

    for t in suspeitas:
        data_formatada = t["data"].strftime("%Y-%m-%d")
        valor_formatado = conversor_moeda(t["valor"])
        print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {data_formatada} | Valor: R$ {valor_formatado}")

###Conversor para formato Brasil

In [ ]:
def conversor_moeda(valor):
    texto_americano = f"{valor:,.2f}"
    formato_brasileiro = texto_americano.replace(",", "X").replace(".", ",").replace("X", ".")

    return formato_brasileiro

###Gerar relatório

In [ ]:
def gerar_relatorio(transacoes):
    if not transacoes:
        return None

    resumo_mensal = {}

    for t in transacoes:
        mes = t["data"].strftime("%Y-%m-%d")[:7]
        valor = t["valor"]
        tipo = t["tipo"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": valor,
                "menor_valor": valor,
                "valores_lista": []
            }

        dados = resumo_mensal[mes]
        dados["quantidade"] += 1
        dados["valores_lista"].append(valor)

        if tipo == "credito":
            dados["total_credito"] += valor
        elif tipo == "debito":
            dados["total_debito"] += valor

        if valor > dados["maior_valor"]:
            dados["maior_valor"] = valor
        if valor < dados["menor_valor"]:
            dados["menor_valor"] = valor

    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = sum(dados["valores_lista"]) / dados["quantidade"]
        del dados["valores_lista"]

    return resumo_mensal

###Exibir relatório

In [ ]:
def exibir_relatorio(resumo_mensal, transacoes, data_antiga, data_recente):
    if not resumo_mensal:
        print("Nenhum dado para exibir.")
        return

    dias_passados = (data_recente - data_antiga).days

    print(f"Período analisado: {data_antiga.strftime('%Y-%m-%d')} → {data_recente.strftime('%Y-%m-%d')} ({dias_passados} dias)")
    print("\n===== RELATÓRIO MENSAL =====")


    for mes, dados in resumo_mensal.items():
        print(f"\nMês: {mes}")
        print(f"  Transações: {dados['quantidade']}")
        print(f"  Total crédito: R$ {conversor_moeda(dados['total_credito'])}")
        print(f"  Total débito:  R$ {conversor_moeda(dados['total_debito'])}")
        print(f"  Saldo:         R$ {conversor_moeda(dados['saldo'])}")
        print(f"  Média:         R$ {conversor_moeda(dados['media'])}")
        print(f"  Maior valor:   R$ {conversor_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   R$ {conversor_moeda(dados['menor_valor'])}")


    transacoes_suspeitas(transacoes)

###Salvar JSON

In [ ]:
import json

def salvar_json(resumo_mensal, total_validas, total_invalidas, caminho_saida="relatorio.json"):
    data_geracao = datetime.now().strftime("%Y-%m-%d")

    dados_json = {
        "gerado_em": data_geracao,
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": resumo_mensal
    }

    try:
        with open(caminho_saida, mode="w", encoding="utf-8") as arquivo:
            json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)
        print(f"\n[Sucesso] Relatório exportado com sucesso para: {caminho_saida}")
    except IOError as e:
        print(f"Erro ao salvar o arquivo JSON: {e}")

##Célula principal

In [ ]:
CAMINHO_CSV = "/content/drive/MyDrive/Estudo Python - Pós em IA/Desafio/transacoes.csv"
linhas_brutas = ler_transacoes(CAMINHO_CSV)

if linhas_brutas is not None:
    transacoes_validas = []
    linhas_invalidas = 0

    for linha in linhas_brutas:
        registro_limpo = validar_transacao(linha)
        if registro_limpo is not None:
            transacoes_validas.append(registro_limpo)
        else:
            linhas_invalidas += 1


    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {len(linhas_brutas)}")
    print(f"Linhas válidas: {len(transacoes_validas)}")
    print(f"Linhas inválidas: {linhas_invalidas}\n")


    resumo_calculado = gerar_relatorio(transacoes_validas)

    if resumo_calculado:
        datas = [t["data"] for t in transacoes_validas]
        data_antiga = min(datas)
        data_recente = max(datas)

        exibir_relatorio(resumo_calculado, transacoes_validas, data_antiga, data_recente)

        salvar_json(resumo_calculado, len(transacoes_validas), linhas_invalidas)

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5

Período analisado: 2026-01-05 → 2026-03-25 (79 dias)

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 16.000,00
  Total débito:  R$ 345,50
  Saldo:         R$ 15.654,50
  Média:         R$ 3.269,10
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 45,00

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 15.500,00
  Total débito:  R$ 659,90
  Saldo:         R$ 14.840,10
  Média:         R$ 3.231,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 3.950,00
  Total débito:  R$ 1.449,90
  Saldo:         R$ 2.500,10
  Média:         R$ 1.079,98
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90

===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cliente: CLI004 | Data: 2026-01-22 | Valor: R$ 12.500,00
ID: 8 | Cliente: CLI005 | Data: 2026-02-14 | Valor: R$ 15.000,00

[Sucesso] Relatório exportado com sucesso para: rel